*0.2 Math / ML basics*

# sampling: min-p

**The situation.** Creative writing at temperature 1.3 with top-p 0.95 gives lively text with occasional garbage. Lowering temperature kills the liveliness; lowering top-p cuts real options when the model is open-ended. Neither knob targets the actual problem: tokens that are *much* worse than the best one.

**Min-p.** Look at the top token's probability. Keep every token whose probability is at least *min_p* times that. With min_p = 0.1 and a top token at 60%, anything under 6% is dropped; with a top token at 5%, anything under 0.5% is dropped. The threshold scales with the model's confidence. Supported by Ollama, vLLM and llama.cpp; not by OpenAI.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch


def min_p_keep(probabilities: torch.Tensor, min_p: float) -> int:
    threshold = probabilities.max() * min_p
    return int((probabilities >= threshold).sum())


certain = torch.softmax(torch.tensor([8.0, 2.0, 1.0, 0.5, 0.0]), dim=0)
open_ended = torch.softmax(torch.tensor([1.0, 1.0, 0.9, 0.9, 0.8]), dim=0)
for name, distribution in (("certain", certain), ("open-ended", open_ended)):
    threshold = float(distribution.max() * 0.1)
    print(
        
            f"{name:<11} top token {distribution.max():.2f} → threshold {threshold:.3f} → keeps "
            f"{min_p_keep(distribution, 0.1)} of 5"
        
    )
assert min_p_keep(certain, 0.1) == 1 and min_p_keep(open_ended, 0.1) == 5

certain     top token 1.00 → threshold 0.100 → keeps 1 of 5
open-ended  top token 0.22 → threshold 0.022 → keeps 5 of 5


**Reading the output.** Like top-p, min-p kept 1 token when the model was certain and all 5 when it was open — but the threshold was set *relative to the best token*, so a long flat tail of "much worse" tokens is cut even when the distribution is open.

**For real, on the local model.** min_p 0.5 (strict) vs 0.0 (off) at temperature 1.5.

In [3]:
from ollama import Client

ollama = Client()
for min_p in (0.5, 0.0):
    seen = set()
    for sample in range(5):
        reply = ollama.generate(
            model="qwen2.5:0.5b",
            prompt="Name one colour. Answer with one word.",
            options={
                "min_p": min_p,
                "top_p": 1.0,
                "top_k": 0,
                "temperature": 1.5,
                "num_predict": 3,
                "seed": sample + 1,
            },
        )
        seen.add(reply["response"].strip().lower().strip("."))
    print(f"min_p={min_p}: {len(seen)} distinct answer(s) → {sorted(seen)}")

min_p=0.5: 1 distinct answer(s) → ['red']
min_p=0.0: 3 distinct answer(s) → ['red', 'yellow', '赤とした spe']


```
top token 0.60  → keep ≥ 0.06     (min_p = 0.1)
top token 0.05  → keep ≥ 0.005    threshold follows confidence
```

**The rule to remember.** Min-p cuts tokens that are much worse than the best one, whatever the shape. 0.05–0.1 is the common range; it pairs well with higher temperatures.

| Use it when | Don't when | Instead use |
|---|---|---|
| open-source serving, creative tasks at high temperature | hosted APIs that do not support it | top-p |

**Watch out**
- Turn top-p and top-k off (1.0 and 0) when testing min-p, or three filters stack and you cannot tell which one acted.
- min_p = 0.5 is very strict — almost greedy. Start at 0.05.
- Newer than the others; check your serving stack's version before relying on it.